### **Reading Dataset**

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Load Data
df_reviews = pd.read_csv('Amazon_Reviews.csv', engine='python', on_bad_lines='skip')

In [ ]:
rating_col = 'Rating'

df_reviews = df_reviews.dropna(subset=[rating_col]).copy() #drop missing/empty values

# convert string to numeric digit (eg: "Rated 1 out of 5 stars" -> 1)
df_reviews['numeric_rating'] = df_reviews[rating_col].str.extract(r'(\d+)').astype(int)

#print(df_reviews['numeric_rating'])

# Set 1-2 to negative, 3 to neutral & 4-5 to positive
def categorize_rating(rating):
    if rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'

df_reviews['sentiment'] = df_reviews['numeric_rating'].apply(categorize_rating) #.apply: loop

#print(df_reviews[[rating_col, 'numeric_rating', 'sentiment']].head(20))

In [ ]:
# Check total available reviews per sentiment
print(df_reviews['sentiment'].value_counts())

In [ ]:
# Keep all positive and negative reviews
df_reviews_amazon = df_reviews[
    df_reviews['sentiment'].isin(['positive', 'negative'])
].copy()

print(df_reviews_amazon['sentiment'].value_counts())

# **Text Preprocessing**

In text preproccessing, i just want to apply some standard way of preprocessing in sentiment analysis, in order to make the model understand it better. These step are lowercase, clear punctuation, stopword, and stemming. You can experience with any other step to see what combination might lead to a better result.

In [ ]:
# TESTING
# !pip install contractions
# import contractions

# expanded_text = "Don't buy goods from amazon!!!"

# # Use contractions.fix() directly on the string
# text = contractions.fix(expanded_text, slang=False)

# print(text)

In [ ]:
!pip install contractions
import contractions
import re

In [ ]:
## 1. Cleaning
def clean_text(text):
  # Expand contractions (don't -> do not)
  text = contractions.fix(text, slang=False)

  # Lowercase
  text = text.lower()

  # # Clear punctuation
  # text = re.sub('\[.*?\]','',text)

  # Remove URL
  text = re.sub(r'https?://\S+|www\.\S+', '', text)

  # Remove HTML tags
  text = re.sub(r'<.*?>', ' ', text)

  # Remove emojis and special characters (keep only letters, NUMBERS & spaces)
  text = re.sub(r'[^a-z0-9\s]', ' ', text)

  # Remove extra whitespace
  text = " ".join(text.split())

  return text
# TESTING
# raw_input = "www.amazon/      in/gp/product/B07NDT6PLC/ref=ppx_yo_dt_b_asin_title_o00_s00?ie=UTF8&psc=1Whirlpool 1 Ton 5 Star Inverter Split AC (Alloy, 1.0T 3DCOOL INVERTER 5S-W, White)Purchased the product on amazon. Import date is Dce 2017.I ordered this on 8th of march delivered on 10th. As the date mentioned first available from 5 February 2019 considering this is 2019 product. As no questions and reviews were there in this product listing. Raised return request.  Customer care called me thrice today and they were only keep saying sorry. They did not ready to accept return request. They are accepting the fact that they are selling old garbage products. But keep pointing to their policies.. Never buy .."
# cleaned_text = clean_text(raw_input)
# print(cleaned_text)


In [ ]:
df_reviews_amazon['Review Text'] = df_reviews_amazon['Review Text'].apply(clean_text)
#print(df_reviews_amazon['Review Text'].head())

In [ ]:
# ## 2. Stopword
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

# # 1. Custom stop words list preserving negations
# custom_stop_words = list(ENGLISH_STOP_WORDS)
# for word in ['not', 'no', 'never', 'nor']:
#     if word in custom_stop_words:
#         custom_stop_words.remove(word)

# # 2. Advanced TF-IDF (keeps unigrams + bigrams + negations)
# tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words=custom_stop_words)
# #ngram_range: to look at single words & pairs of words because we have words like "right shambles, & our model treated it as +ve review"

# # 3. Transform training and testing text sets
# train_x_vector = tfidf.fit_transform(train_x)
# test_x_vector = tfidf.transform(test_x)

In [ ]:
# 3. Lemmatization
# import nltk
# from nltk.stem import WordNetLemmatizer
# from nltk.corpus import wordnet

# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger_eng')
# nltk.download('punkt_tab')

# lemmatizer = WordNetLemmatizer()

# def get_wordnet_pos(tag):
#     tag_map = {
#         'J': wordnet.ADJ,
#         'V': wordnet.VERB,
#         'N': wordnet.NOUN,
#         'R': wordnet.ADV
#     }
#     return tag_map.get(tag[0].upper(), wordnet.NOUN)

# def lemmatize_text(text):
#     tokens = nltk.word_tokenize(text)
#     tagged_tokens = nltk.pos_tag(tokens)

#     lemmas = [
#         lemmatizer.lemmatize(word, get_wordnet_pos(tag))
#         for word, tag in tagged_tokens
#     ]

#     return " ".join(lemmas)

# df_reviews_amazon['Review Text'] = df_reviews_amazon['Review Text'].apply(lemmatize_text)


# CAN REMOVE: Apply cleaning before tokenization
# train_df['text'] = train_df['text'].apply(clean_review_text)
# test_df['text'] = test_df['text'].apply(clean_review_text

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# df_reviews_amazon[['Review Text', 'clean_text']].head(5)

# **Splitting data into train and test**

In [ ]:
# import train_test_split
from sklearn.model_selection import train_test_split

# split imbalanced DataFrame (60% train, 20% validation, 20% test)
# Step 1: Split off 20% for the test set
train_val, test = train_test_split(
    df_reviews_amazon,
    test_size=0.20,
    random_state=42,
    stratify=df_reviews_amazon['sentiment']
)

# Step 2: Split the remaining 80% into training and validation
# 25% of 80% = 20% of the original dataset
train, val = train_test_split(
    train_val,
    test_size=0.25,
    random_state=42,
    stratify=train_val['sentiment']
)

In [ ]:
# Separate the review text (X) and labels (y)
train_x, train_y = train['Review Text'], train['sentiment']
val_x, val_y = val['Review Text'], val['sentiment']
test_x, test_y = test['Review Text'], test['sentiment']

In [ ]:
# Check the number of rows
print("Training:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

Training: 12102
Validation: 4034
Test: 4034


# **Dealing with Imbalanced Classes**

In [ ]:
# # Before: see how data is distributed
# df_reviews_amazon.value_counts('sentiment').plot(kind='bar')

In [ ]:
# # take a sample of 5000 rows to make processing faster and get imbalance data
# # 4000 positives
# df_positive = df_reviews[df_reviews['sentiment']=='positive'][:4000]
# # 1000 negative
# df_negative = df_reviews[df_reviews['sentiment']=='negative'][:1000]

In [ ]:
# # Combine
# df_reviews_amazon = pd.concat([df_positive, df_negative])

In [ ]:
# df_reviews_amazon.value_counts('sentiment')

In [ ]:
# # After: see how data is distributed
# df_reviews_amazon.value_counts('sentiment').plot(kind='bar')

### **Option A. Balancing data with .sample()**
#### Balance data manually ####

In [ ]:
# length_negative = len(df_reviews_amazon[df_reviews_amazon['sentiment']=='negative'])

In [ ]:
# randomly picks 1,000 positive reviews
# df_reviews_positive = df_reviews_amazon[df_reviews_amazon['sentiment']=='positive'].sample(n=length_negative)

In [ ]:
# df_reviews_negative = df_reviews_amazon[df_reviews_amazon['sentiment']=='negative']

In [ ]:
# df_reviews_bal_manual = pd.concat([df_reviews_positive, df_reviews_negative])

In [ ]:
# # reset the index
# df_reviews_bal_manual.reset_index(drop=True, inplace=True)

In [ ]:
# df_reviews_bal_manual.value_counts('sentiment')

### **Option B. Balancing data with RandomUnderSampler (This option is chosen)** ###
#### run RandomUnderSampler using df_reviews_amazon (orginal dataset) not the manually balanced def_reviews_bal ####

In [ ]:
!pip install imbalanced-learn

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)

train_x_balanced, train_y_balanced = rus.fit_resample(
    train_x.to_frame(),
    train_y
)

train_x_balanced = train_x_balanced['Review Text']

print(train_y_balanced.value_counts())

sentiment
negative    3492
positive    3492
Name: count, dtype: int64


In [ ]:
#  df_reviews_bal.value_counts('sentiment')

# **Text Representation (Bag of Words)**

### **Count Vectorizer** (testing)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

text = ["I love writing code in Python. I love Python code",
        "I hate writing code in Java. I hate Java code"]
df = pd.DataFrame({'review':['review1', 'review2'], 'text':text})

cv = CountVectorizer(stop_words='english')
cv_matrix = cv.fit_transform(df['text'])

In [ ]:
df_dtm = pd.DataFrame(cv_matrix.toarray(), index=df['review'].values, columns=cv.get_feature_names_out())

In [ ]:
df_dtm

,code,hate,java,love,python,writing
review1,2,0,0,2,2,1
review2,2,2,2,0,0,1



### **Tfidf**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

text = ["I love writing code in Python. I love Python code",
        "I hate writing code in Java. I hate Java code"]

df = pd.DataFrame({'review':['review1', 'review2'], 'text':text})
tfidf = TfidfVectorizer(stop_words='english', norm=None)
tfidf_matrix = tfidf.fit_transform(df['text'])

In [ ]:
pd.DataFrame(tfidf_matrix.toarray(), index=df['review'].values,
                      columns=tfidf.get_feature_names_out())

,code,hate,java,love,python,writing
review1,2.0,0.00000,0.00000,2.81093,2.81093,1.0
review2,2.0,2.81093,2.81093,0.00000,0.00000,1.0


### **Turning our text data into numerical vectors**

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# tfidf = TfidfVectorizer(stop_words='english')
# train_x_vector = tfidf.fit_transform(train_x)

In [ ]:
## #I REPLACE THIS
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

# 1. Custom stop words list preserving negations
custom_stop_words = list(ENGLISH_STOP_WORDS)

for word in ['not', 'no', 'never', 'nor']:
    if word in custom_stop_words:
        custom_stop_words.remove(word)

# 2. Advanced TF-IDF (keeps unigrams + bigrams + negations)
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words=custom_stop_words)
#ngram_range: to look at single words & pairs of words because we have words like "right shambles, & our model treated it as +ve review"

# Fit TF-IDF using balanced training reviews only
train_x_vector = tfidf.fit_transform(train_x_balanced)

# Transform validation and test reviews using the same TF-IDF
val_x_vector = tfidf.transform(val_x)
test_x_vector = tfidf.transform(test_x)

print("Training:", train_x_vector.shape)
print("Validation:", val_x_vector.shape)
print("Test:", test_x_vector.shape)

Training: (6984, 127349)
Validation: (4034, 127349)
Test: (4034, 127349)


In [ ]:
test_x_vector = tfidf.transform(test_x)

In [ ]:
train_x_vector

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 378735 stored elements and shape (6984, 127349)>

In [ ]:
test_x_vector

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 191206 stored elements and shape (4034, 127349)>

In [ ]:
pd.DataFrame.sparse.from_spmatrix(
    train_x_vector,
    index=train_x_balanced.index,
    columns=tfidf.get_feature_names_out()
)

,00,00 00,00 10,00 18,00 21,00 30,00 86,00 95,00 account,00 aedorder,...,zone,zone customer,zone slipper,zone store,zoom,zoom barcode,zou,zou denken,zoverstocks,zoverstocks receive
12377,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2122,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8939,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12706,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
11598,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19747,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13883,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13593,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
18795,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# **Model Selection**

### **Support Vector Machines (SVM)**

In [ ]:
from sklearn.svm import SVC

svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y_balanced)

In [ ]:
print(svc.predict(tfidf.transform(['Many products and great prices! Decent speed in delivery. Will purchase from the again!'])))
print(svc.predict(tfidf.transform(['Item arrived broken and customer support was okay.'])))
print(svc.predict(tfidf.transform(['Right shambles'])))
print(svc.predict(tfidf.transform(['This product is not good and I do not recommend it.'])))
print(svc.predict(tfidf.transform(['Why do I say “don’t buy from Amazon”? On July 25th I ordered from Amazon UK a German electric razor. I got an email from Ninth Avenue taking the money and saying I could monitor the order at an R.R.Donnelley number. There was no info at this number. In August I complained that no razor had been delivered. They apologised for this “inconvenience”. Later I got a US mailed sealed envelope with an empty razor case. They ignored when I emailed this fact. They said the goods had been sent from a US warehouse and I should ask the local post and tell them the results. I told them I would be laughed out of it if I asked what happened to goods sent from an unknown US depot on a date unknown. I asked many times for my money back. I am still asking. Their ‘support service’ Headrush emailed that they had now satisfied my query which they were closing. As of 15 November I have received no razor and no money.'])))
print(svc.predict(tfidf.transform(['Looks good but to be honest pretty terrible. The delivery time is terrible and the price is way too expensive. Ebay is way better. '])))
print(svc.predict(tfidf.transform(['Diff to contact / cant solve simplest of issues'])))
#print(svc.predict(tfidf.transform(['I ordered a knife on Amazon but it cames late and I don't need it anymore. Amazon refunded me without asking me to return it. Can't complain.'])))

### **Decision Tree Classifier**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dec_tree = DecisionTreeClassifier(random_state=42)
dec_tree.fit(train_x_vector, train_y_balanced)

### **Gaussian Naive Bayes (not gaussian anymore!)**

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(train_x_vector, train_y_balanced)

### **Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(train_x_vector, train_y_balanced)

**1. Tokens=256**:

--- 1 - DistilBERT Confusion Matrix  ---
[[313  12]
 [ 27 308]]

--- 2 - DistilBERT Predictions - Classification Report:---
              precision    recall  f1-score   support

    negative           0.92      0.96      0.94       325
    positive           0.96      0.92      0.94       335

    accuracy                               0.94       660
    macro avg          0.94      0.94      0.94       660
    weighted avg       0.94      0.94      0.94       660

**2. num_train_epochs=2 -> 4;
     learning_rate=2e-5 -> 3e-5;
     **:

--- 1 - DistilBERT Confusion Matrix  ---
[[307  18]
 [ 22 313]]

--- 2 - DistilBERT Predictions - Classification Report:---
              precision    recall  f1-score   support

    negative           0.93      0.94      0.94       325
    positive           0.95      0.93      0.94       335

    accuracy                               0.94       660
    macro avg          0.94      0.94      0.94       660
    weighted avg       0.94      0.94      0.94       660

# **Model Evaluation**

###**Comparing Models**

In [ ]:
from sklearn.metrics import classification_report

models = {
    'SVM': svc,
    'Decision Tree': dec_tree,
    'Naive Bayes': nb,
    'Logistic Regression': log_reg
}

for name, model in models.items():
    val_predictions = model.predict(val_x_vector)

    print(f"\n--- {name} ---")
    print(classification_report(val_y, val_predictions))


###**SVM - Confusion Matrix**

In [ ]:
# calculate confusion matrix of svc
from sklearn.svm import SVC

svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y_balanced)


### **All models - Mean Accuracy**

In [ ]:
# calculate accuracy of all models

print(svc.score(test_x_vector, test_y))
print(dec_tree.score(test_x_vector, test_y))
print(nb.score(test_x_vector, test_y))
print(log_reg.score(test_x_vector, test_y))

# model.score(test_x_vector, test_y)

###**SVM - F1 Score**

In [ ]:
from sklearn.metrics import f1_score

f1_score(test_y, svc.predict(test_x_vector),
         labels=['positive', 'negative'],
         average=None)


### **SVM - Classification report**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                      svc.predict(test_x_vector),
                      labels=['positive', 'negative']))


### **EXTRA MODEL - DistilBERT Model**

In [ ]:
!pip install -q transformers[torch] datasets

In [ ]:
# # ========================
# # EXTRA MODEL: DistilBERT
# # ========================
# import torch
# import numpy as np
# import pandas as pd
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# from datasets import Dataset
# from sklearn.metrics import classification_report, accuracy_score, f1_score

# print("--- Step 1: Preparing Data for DistilBERT ---")

# # 1. Convert sentiments ('negative'/'positive') to integers (0/1)
# label_map = {'negative': 0, 'positive': 1}
# y_train_num = [label_map[label] for label in train_y]
# y_test_num = [label_map[label] for label in test_y]

# # 2. Convert Pandas Series to Hugging Face Dataset format
# train_df = pd.DataFrame({'text': train_x, 'label': y_train_num})
# test_df = pd.DataFrame({'text': test_x, 'label': y_test_num})
# # ------------------------------------------------------------------------- #
# # Clean text BEFORE converting to Hugging Face Dataset
# # Clean it because reviews might contain HTML code left behind by web scrapers and messy user formatting
# import re

# def clean_review_text(text):
#     text = re.sub(r'<.*?>', ' ', text)  # Remove HTML tags like <br />
#     text = re.sub(r'\s+', ' ', text)    # Replace multiple spaces with a single space
#     return text.strip()

# # Apply cleaning before tokenization
# train_df['text'] = train_df['text'].apply(clean_review_text)
# test_df['text'] = test_df['text'].apply(clean_review_text)
# # -------------------------------------------------------------------------- #

# # 3. Load Pretrained Tokenizer
# tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# def tokenize_func(examples):
#     return tokenizer(examples['text'], truncation=True, max_length=256, padding="max_length") # Cuts off any review > than 256 tokens

# tokenized_train = train_ds.map(tokenize_func, batched=True)
# tokenized_test = test_ds.map(tokenize_func, batched=True)

# # 4. Load Pretrained Model & move to GPU
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Using device: {device.upper()}")

# bert_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# # 5.
# # During Training - compute metrics
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     preds = np.argmax(logits, axis=1)
#     acc = accuracy_score(labels, preds)
#     f1 = f1_score(labels, preds, average="weighted")
#     return {"accuracy": acc, "f1": f1}



# # 6. Configure Training Arguments
# training_args = TrainingArguments(
#     output_dir="./distilbert_amazon",
#     eval_strategy="epoch",
#     learning_rate=3e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=4, # Model sees the data 4 times
#     weight_decay=0.01,
#     logging_steps=20,
#     report_to="none"  # Disables external logging like WandB
# )

# # 7. Initialize Trainer
# trainer = Trainer(
#     model=bert_model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_test,
#     compute_metrics=compute_metrics,
# )

# print("\n--- Step 2: Training DistilBERT ---")
# trainer.train()

# # 8. Evaluate and Print Classification Report
# predictions = trainer.predict(tokenized_test)
# bert_preds = np.argmax(predictions.predictions, axis=1)

# print("\n--- Step 3: Evaluating DistilBERT Prediction  ---")
# # Confusion Matrix
# print("\n--- 1 - DistilBERT Confusion Matrix  ---")
# print(confusion_matrix(y_test_num, bert_preds))

# # Classification Report
# print("\n--- 2 - DistilBERT Predictions - Classification Report:---")
# print(classification_report(y_test_num, bert_preds, target_names=['negative', 'positive']))